# Call Center Agent Patient Data RAG Pipeline
This notebook provides a RAG-based multi-agent search for call center representatives assisting patients.

In [ ]:
!pip install -q openai litellm gradio

In [ ]:
import os
from openai import OpenAI
import sys
sys.path.append(os.getcwd()) # Ensure local agent module imports correctly
from agents import DataRetrievalAgent, SummaryAgent

In [ ]:
# OpenRouter API Setup
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Warning: OPENAI_API_KEY environment variable not set. RAG pipeline will run in mock mode.")

try:
    client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1") if api_key else None
    print("LLM Client Ready.")
except Exception as e:
    print(f"Error initializing client: {e}")
    client = None

In [ ]:
# Initialize Agents
retriever = DataRetrievalAgent()
summarizer = SummaryAgent(use_llm=True, client=client)
print("Agents initialized.")

In [ ]:
def fetch_patient_profile(query: str) -> str:
    """Pipeline runner: Retrieves raw data then summarizes it."""
    raw_data = retriever.retrieve(query)
    summary = summarizer.summarize(raw_data)
    return summary

# Test it locally
print(fetch_patient_profile("John Doe"))

In [ ]:
import gradio as gr

def gradio_interface(query: str) -> str:
    if not query.strip():
        return "Please enter a patient ID or name."
    return fetch_patient_profile(query)

app = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(
        label="Enter Patient Name or ID",
        placeholder="e.g., John Doe or P001",
        lines=1
    ),
    outputs=gr.Textbox(label="Patient Summary for Call Center Agent", lines=15),
    title="Call Center - Patient Profile Retrieval System",
    description="A mock RAG pipeline to pull demographics, insurance, hospital stays, visits, and claims using multi-agent logic."
)

app.launch(share=False)